In [1]:
from optilab.aliases import Model, GRB, quicksum
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# ---------------------------------------------------------------------
# Helper: solve assignment model
# ---------------------------------------------------------------------
def solve_cloud_assignment(n_agents, n_tasks, capacity_mode="balanced", seed=1):
    rng = np.random.default_rng(seed)

    m = Model()

    # -----------------------------------------------------------------
    # Problem data (cloud infrastructure)
    # -----------------------------------------------------------------
    cost = rng.integers(1, 20, size=(n_agents, n_tasks))

    # -----------------------------------------------------------------
    # Decision variables
    # -----------------------------------------------------------------
    x = [
        [m.add_binary_var(name=f"x_{i}_{j}") for j in range(n_tasks)]
        for i in range(n_agents)
    ]

    # -----------------------------------------------------------------
    # Objective: minimize allocation cost
    # -----------------------------------------------------------------
    m.set_objective(
        quicksum(cost[i][j] * x[i][j]
                 for i in range(n_agents)
                 for j in range(n_tasks)),
        GRB.MINIMIZE
    )

    # -----------------------------------------------------------------
    # Constraints
    # -----------------------------------------------------------------

    if capacity_mode == "balanced":
        # one-to-one structure (idealized cloud system)

        for i in range(n_agents):
            m.add_constraint(
                quicksum(x[i][j] for j in range(n_tasks)) == 1
            )

        for j in range(n_tasks):
            m.add_constraint(
                quicksum(x[i][j] for i in range(n_agents)) == 1
            )

    elif capacity_mode == "unbalanced":
        # realistic cloud system: more tasks than agents

        # each task must be assigned at most once
        for j in range(n_tasks):
            m.add_constraint(
                quicksum(x[i][j] for i in range(n_agents)) <= 1
            )

        # limited compute capacity per agent
        for i in range(n_agents):
            m.add_constraint(
                quicksum(x[i][j] for j in range(n_tasks)) <= 3
            )

    else:
        raise ValueError("capacity_mode must be 'balanced' or 'unbalanced'")

    # -----------------------------------------------------------------
    # Solve
    # -----------------------------------------------------------------
    m.solve()

    solution = np.array([[x[i][j].x for j in range(n_tasks)]
                         for i in range(n_agents)])

    return solution, cost